Сделано с асгардархеей.

In [5]:
!pip install Bio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 321.3/321.3 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 3.4 MB/s eta 0:00:00


In [18]:
from Bio import SeqIO
import numpy as np
import pandas as pd

records = list(SeqIO.parse("gen.fna", "fasta"))
print(f"Находим {len(records)} последовательностей в файле")

seq = ""
for record in records:
    seq += str(record.seq).upper()

print(f"Полная длина: {len(seq)}")

nucleotides = ['A', 'C', 'G', 'T']

dinucleotide_counts = {i+j: 0 for i in nucleotides for j in nucleotides}

for i in range(len(seq) - 1):
    dinucl = seq[i:i+2]
    if dinucl in dinucleotide_counts:
        dinucleotide_counts[dinucl] += 1

transition_matrix = np.zeros((4, 4))

for i, char_i in enumerate(nucleotides):
    row_sum = 0
    for j, char_j in enumerate(nucleotides):
        count = dinucleotide_counts[char_i + char_j]
        transition_matrix[i, j] = count
        row_sum += count

    if row_sum > 0:
        transition_matrix[i, :] /= row_sum

df_tm = pd.DataFrame(transition_matrix, index=nucleotides, columns=nucleotides)
print("\nМатрица переходов (P):")
print(df_tm)

row_sums = transition_matrix.sum(axis=1)
print(f"\nСуммы строк (должны быть 1.0): {row_sums}")

eigenvalues, eigenvectors = np.linalg.eig(transition_matrix.T)
idx = np.argmin(np.abs(eigenvalues - 1.0))
stationary = np.real(eigenvectors[:, idx])
stationary = stationary / stationary.sum()

print("\nСтационарное распределение (π):")
for n, val in zip(nucleotides, stationary):
    print(f"{n}: {val:.4f}")

observed_freqs = {n: seq.count(n) / len(seq) for n in nucleotides}
print("\nНаблюдаемые частоты нуклеотидов:")
for n in nucleotides:
    print(f"{n}: {observed_freqs[n]:.4f}")

Находим 295 последовательностей в файле
Полная длина: 1827468

Матрица переходов (P):
          A         C         G         T
A  0.365928  0.136469  0.175304  0.322299
C  0.348024  0.180313  0.141289  0.330373
G  0.362255  0.202829  0.179245  0.255671
T  0.258470  0.192450  0.183766  0.365314

Суммы строк (должны быть 1.0): [1. 1. 1. 1.]

Стационарное распределение (π):
A: 0.3271
C: 0.1738
G: 0.1728
T: 0.3262

Наблюдаемые частоты нуклеотидов:
A: 0.3271
C: 0.1738
G: 0.1728
T: 0.3262


Наблюдаемые частоты совпадают со стационарным распределением. Это означает, что локальные правила переходов (динуклеотидный состав) полностью объясняют глобальный нуклеотидный состав данной ДНК. Последовательность является однородной с точки зрения марковских процессов первого порядка.